# Xử lý câu mô tả của bệnh nhân (NLP)

Notebook **riêng** này nhận câu mô tả tiếng Việt tự do của bệnh nhân
(`input_mota_benhnhan.csv`) và **tách từ → nhận ra các triệu chứng**,
sau đó chuyển thành vector số giống như dữ liệu huấn luyện.


## Bước 1. Đọc dữ liệu input và danh sách triệu chứng

In [1]:
import pandas as pd

inp = pd.read_csv("../01_data/input_mota_benhnhan.csv")
trongso = pd.read_csv("../01_data/trongso_mucdo_nghiemtrong.csv")
trong_so = dict(zip(trongso["TrieuChung"], trongso["TrongSo"]))
ds_trieuchung = list(trongso["TrieuChung"])
inp


,id,mo_ta
0,1,"Mấy hôm nay tôi sốt cao, đau nhức cơ và mệt mỏ..."
1,2,"Tôi bị hắt hơi liên tục, sổ mũi và nghẹt mũi, ..."
2,3,"Bệnh nhân sốt cao, nổi phát ban, đau sau hốc m..."
3,4,"Tôi ho có đờm nhiều, khó thở và đau tức ngực"
4,5,"Đau bụng quặn từng cơn, tiêu chảy nhiều lần kè..."
5,6,"Đau đầu dữ dội một bên, sợ ánh sáng, buồn nôn ..."


## Bước 2. Tách từ (tokenize)

Tách câu thành các từ. Nếu máy có cài `underthesea` (thư viện NLP tiếng Việt)
thì tách chuẩn hơn; nếu không có thì tách đơn giản theo khoảng trắng.


In [2]:
import re

def chuan_hoa(text):
    text = str(text).lower()
    text = re.sub(r"[.,;!?()]", " ", text)   # bỏ dấu câu
    text = re.sub(r"\s+", " ", text).strip()
    return text

try:
    from underthesea import word_tokenize
    def tach_tu(text):
        return word_tokenize(chuan_hoa(text))
    print("Dùng underthesea để tách từ.")
except Exception:
    def tach_tu(text):
        return chuan_hoa(text).split()
    print("Không có underthesea -> tách theo khoảng trắng.")

inp["tu"] = inp["mo_ta"].apply(tach_tu)
inp[["mo_ta", "tu"]]


Không có underthesea -> tách theo khoảng trắng.


,mo_ta,tu
0,"Mấy hôm nay tôi sốt cao, đau nhức cơ và mệt mỏ...","[mấy, hôm, nay, tôi, sốt, cao, đau, nhức, cơ, ..."
1,"Tôi bị hắt hơi liên tục, sổ mũi và nghẹt mũi, ...","[tôi, bị, hắt, hơi, liên, tục, sổ, mũi, và, ng..."
2,"Bệnh nhân sốt cao, nổi phát ban, đau sau hốc m...","[bệnh, nhân, sốt, cao, nổi, phát, ban, đau, sa..."
3,"Tôi ho có đờm nhiều, khó thở và đau tức ngực","[tôi, ho, có, đờm, nhiều, khó, thở, và, đau, t..."
4,"Đau bụng quặn từng cơn, tiêu chảy nhiều lần kè...","[đau, bụng, quặn, từng, cơn, tiêu, chảy, nhiều..."
5,"Đau đầu dữ dội một bên, sợ ánh sáng, buồn nôn ...","[đau, đầu, dữ, dội, một, bên, sợ, ánh, sáng, b..."


## Bước 3. Nhận diện triệu chứng trong câu

Tên triệu chứng của ta là **cụm tiếng Việt** (ví dụ: `"sốt cao"`, `"ho có đờm"`),
nên ta tìm trực tiếp cụm đó trong câu. Ta thêm vài **từ đồng nghĩa** để bắt được
cách diễn đạt khác nhau. Ưu tiên khớp cụm **dài trước** (để "sốt cao" được nhận
trước "sốt").


In [3]:
# Một vài cách nói khác -> tên triệu chứng chuẩn
dong_nghia = {
    "tức ngực": "đau ngực",
    "đau tức ngực": "đau ngực",
    "sợ ánh sáng": "nhạy cảm ánh sáng",
    "đau mỏi cơ": "đau nhức cơ",
}

def tim_trieu_chung(text):
    norm = chuan_hoa(text)
    tim_thay = []
    # gộp tên triệu chứng chuẩn + từ đồng nghĩa, sắp xếp cụm dài trước
    ung_vien = list(ds_trieuchung) + list(dong_nghia.keys())
    for cum in sorted(ung_vien, key=len, reverse=True):
        if cum in norm:
            chuan = dong_nghia.get(cum, cum)   # quy về tên chuẩn
            if chuan not in tim_thay:
                tim_thay.append(chuan)
            norm = norm.replace(cum, " ")       # xóa phần đã nhận để khỏi trùng
    return tim_thay

inp["trieu_chung"] = inp["mo_ta"].apply(tim_trieu_chung)
inp[["mo_ta", "trieu_chung"]]


,mo_ta,trieu_chung
0,"Mấy hôm nay tôi sốt cao, đau nhức cơ và mệt mỏ...","[đau nhức cơ, sốt cao, mệt mỏi, ớn lạnh, ho]"
1,"Tôi bị hắt hơi liên tục, sổ mũi và nghẹt mũi, ...","[nghẹt mũi, đau họng, hắt hơi, sổ mũi]"
2,"Bệnh nhân sốt cao, nổi phát ban, đau sau hốc m...","[đau sau hốc mắt, đau khớp, phát ban, sốt cao]"
3,"Tôi ho có đờm nhiều, khó thở và đau tức ngực","[đau ngực, ho có đờm, khó thở]"
4,"Đau bụng quặn từng cơn, tiêu chảy nhiều lần kè...","[tiêu chảy, buồn nôn, đau bụng]"
5,"Đau đầu dữ dội một bên, sợ ánh sáng, buồn nôn ...","[nhạy cảm ánh sáng, chóng mặt, buồn nôn, đau đầu]"


## Bước 4. Chuyển thành vector số (one-hot có trọng số) và lưu

Vector này có **cùng thứ tự cột** với dữ liệu huấn luyện, nên có thể đưa thẳng
vào mô hình đã train để dự đoán bệnh.


In [4]:
def thanh_vector(trieu_chung):
    vec = {tc: 0 for tc in ds_trieuchung}
    for tc in trieu_chung:
        if tc in vec:
            vec[tc] = trong_so[tc]
    return vec

vec_df = pd.DataFrame([thanh_vector(t) for t in inp["trieu_chung"]])
vec_df.insert(0, "mo_ta", inp["mo_ta"])

import os
os.makedirs("../01_data/processed", exist_ok=True)
vec_df.to_csv("../01_data/processed/input_vectorized.csv", index=False, encoding="utf-8-sig")
print("Đã lưu: 01_data/processed/input_vectorized.csv")
vec_df.head()


Đã lưu: 01_data/processed/input_vectorized.csv


,mo_ta,sốt cao,sốt,sốt nhẹ,đau đầu,đau nhức cơ,mệt mỏi,ho,ho có đờm,đau họng,...,đầy hơi,chán ăn,đổ mồ hôi,chảy nước mắt,ngứa,nhạy cảm ánh sáng,chóng mặt,đau mặt,giảm khứu giác,thở khò khè
0,"Mấy hôm nay tôi sốt cao, đau nhức cơ và mệt mỏ...",6,0,0,0,3,3,3,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"Tôi bị hắt hơi liên tục, sổ mũi và nghẹt mũi, ...",0,0,0,0,0,0,0,0,4,...,0,0,0,0,0,0,0,0,0,0
2,"Bệnh nhân sốt cao, nổi phát ban, đau sau hốc m...",6,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,"Tôi ho có đờm nhiều, khó thở và đau tức ngực",0,0,0,0,0,0,0,4,0,...,0,0,0,0,0,0,0,0,0,0
4,"Đau bụng quặn từng cơn, tiêu chảy nhiều lần kè...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
